# AI Story Generator

This notebook uses a **generative language model (GPT-2)** from Hugging Face to write a short story from your choices: genre, hero name, and setting.

**How it works:** we give the model the *start* of a story (a prompt built from your choices). The model then predicts one word (token) at a time to continue it.

Run the cells from top to bottom. In Colab: **Runtime → Run all**.

## 1. Install and import libraries

In [ ]:
!pip install -q transformers torch

In [ ]:
import re
import textwrap
from transformers import pipeline, set_seed

## 2. Load the model

GPT-2 is small (about 500 MB) and runs on the free Colab CPU. The first run downloads it, which takes a minute.

In [ ]:
MODEL_NAME = "gpt2"   # try "gpt2-medium" for slightly better stories (slower)
generator = pipeline("text-generation", model=MODEL_NAME)
print("Model loaded:", MODEL_NAME)

## 3. Story settings

Change these values, then run the next cells. In Colab, the form on the right updates them for you.

In [ ]:
genre = "fantasy"      #@param ["fantasy", "sci-fi", "mystery", "fairy tale", "horror"]
hero = "Asha"          #@param {type:"string"}
setting = "an old lighthouse by the sea"   #@param {type:"string"}
length = "medium"      #@param ["short", "medium", "long"]
creativity = 0.9       #@param {type:"slider", min:0.5, max:1.3, step:0.05}
seed = 42              #@param {type:"integer"}

## 4. Build the prompt

Each genre has an opening sentence. The model continues from it, so the opening steers the style of the whole story.

In [ ]:
OPENINGS = {
    "fantasy":    "Long ago, in a land where magic still lived in the rivers, {hero} discovered a glowing key inside {setting}.",
    "sci-fi":     "In the year 2187, {hero} received a strange signal that seemed to come from {setting}.",
    "mystery":    "It was a rainy night when {hero} arrived at {setting} and found that something important had vanished.",
    "fairy tale": "Once upon a time, near {setting}, lived a kind child named {hero} who was braver than anyone knew.",
    "horror":     "The lights flickered as {hero} stepped into {setting}, where nobody had been seen for years.",
}

TOKENS = {"short": 80, "medium": 160, "long": 260}

def build_prompt(genre, hero, setting):
    return OPENINGS[genre].format(hero=hero, setting=setting)

print(build_prompt(genre, hero, setting))

## 5. Generate the story

Key settings that control the output:

| Setting | What it does |
|---|---|
| `temperature` | Higher = more random and creative, lower = safer and more repetitive |
| `top_k`, `top_p` | Limit which words the model may pick at each step |
| `repetition_penalty` | Discourages the model from repeating itself |
| `max_new_tokens` | Controls how long the story is |

In [ ]:
def clean_story(text):
    """Remove odd whitespace and cut off any unfinished last sentence."""
    text = re.sub(r"\s+", " ", text).strip()
    last_end = max(text.rfind("."), text.rfind("!"), text.rfind("?"))
    return text[: last_end + 1] if last_end > 0 else text

def generate_story(genre, hero, setting, length="medium", creativity=0.9, seed=42):
    set_seed(seed)                      # same seed = same story (reproducible)
    prompt = build_prompt(genre, hero, setting)
    result = generator(
        prompt,
        max_new_tokens=TOKENS[length],
        do_sample=True,
        temperature=creativity,
        top_k=50,
        top_p=0.95,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        pad_token_id=50256,
    )
    return clean_story(result[0]["generated_text"])

story = generate_story(genre, hero, setting, length, creativity, seed)
print(textwrap.fill(story, width=90))

## 6. Try other genres

The loop below writes one short story for each genre so you can compare them.

In [ ]:
for g in OPENINGS:
    print(f"=== {g.upper()} ===")
    print(textwrap.fill(generate_story(g, hero, setting, "short", creativity, seed), width=90))
    print()

## 7. Save your story

In [ ]:
filename = f"story_{genre.replace(' ', '_')}_{hero}.txt"
with open(filename, "w", encoding="utf-8") as f:
    f.write(story)
print("Saved:", filename)

# In Colab, this also downloads the file to your computer:
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    pass

## 8. Notes and limitations

- **Model:** GPT-2 is a small, older model, so stories can wander or lose the plot. That is normal for a model this size.
- **Randomness:** the output changes with `seed` and `creativity`. Try different values.
- **Improvements to try:** a bigger model (`gpt2-medium`), a modern instruction-tuned model, few-shot prompts (show the model an example story first), or fine-tuning on a story dataset.
- **Ethics:** language models can produce biased or odd text, so review the output before using it.